# Spark Structured Streaming Join Example

This notebook reads two Kafka streams and joins them using Spark Structured Streaming.

The two producer notebooks send random numbers to:

- `teaching-stream-A`
- `teaching-stream-B`

This notebook joins events when:

1. the number from Stream A equals the number from Stream B, and
2. the two event timestamps are within 10 seconds of each other.

The joined results are printed directly in the notebook output using `foreachBatch`.

In [ ]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, from_json, to_timestamp
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

## Spark And Kafka Configuration

The package configuration follows the same style used in the assignment solution: Spark is started with the Kafka connector packages.

If your Kafka broker is not local, change `HOST_IP`.

In [ ]:
HOST_IP = "10.192.8.146"
KAFKA_BOOTSTRAP_SERVERS = f"{HOST_IP}:9092"

SPARK_PACKAGES = (
    "org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.5,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {SPARK_PACKAGES} pyspark-shell"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Teaching Example - Join Two Kafka Streams")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session started.")

## Define The Expected JSON Schema

Kafka stores each message as bytes. The producers send JSON, so Spark must parse the Kafka value into columns.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType(), nullable=False),
    StructField("stream_id", StringType(), nullable=False),
    StructField("number", IntegerType(), nullable=False),
    StructField("event_time", StringType(), nullable=False)
])

## Helper Function To Read One Kafka Topic

This function:

1. reads a Kafka topic,
2. converts the Kafka `value` from bytes to string,
3. parses the JSON string using the schema,
4. converts `event_time` into a Spark timestamp,
5. applies a watermark so Spark can eventually clear old streaming state.

In [ ]:
def read_number_stream(topic_name):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", topic_name)
        .option("startingOffsets", "latest")
        .option("failOnDataLoss", "false")
        .load()
        .selectExpr("CAST(value AS STRING) AS json_value")
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        .withColumn("event_timestamp", to_timestamp(col("event_time")))
        .withWatermark("event_timestamp", "30 seconds")
    )


stream_a = read_number_stream("teaching-stream-A")
stream_b = read_number_stream("teaching-stream-B")

print("Kafka streams created.")

## Join The Two Streams

The join condition has two parts:

- `a.number = b.number`
- the event from B must occur within 10 seconds before or after the event from A

This is a stream-stream join. Watermarks are important because they tell Spark how long old records should be kept in state while waiting for possible matches.

In [ ]:
joined_stream = (
    stream_a.alias("a")
    .join(
        stream_b.alias("b"),
        expr("""
            a.number = b.number AND
            b.event_timestamp >= a.event_timestamp - interval 10 seconds AND
            b.event_timestamp <= a.event_timestamp + interval 10 seconds
        """),
        "inner"
    )
    .select(
        col("a.number").alias("matched_number"),
        col("a.event_id").alias("event_id_A"),
        col("b.event_id").alias("event_id_B"),
        col("a.event_timestamp").alias("event_time_A"),
        col("b.event_timestamp").alias("event_time_B")
    )
)

## Print Joined Results In The Notebook

The `foreachBatch` output sink lets us run normal batch DataFrame actions on each micro-batch. Here we use `.show()` so that joined records are printed in the notebook output.

Leave this cell running while both producer notebooks are running. Stop it manually when finished.

In [ ]:
def print_joined_batch(batch_df, batch_id):
    count = batch_df.count()
    
    if count == 0:
        print(f"Batch {batch_id}: no joined rows")
    else:
        print(f"Batch {batch_id}: {count} joined row(s)")
        batch_df.orderBy("event_time_A", "event_time_B").show(truncate=False)


query = (
    joined_stream.writeStream
    .foreachBatch(print_joined_batch)
    .outputMode("append")
    .option("checkpointLocation", "checkpoint_stream_join_example")
    .start()
)

print("Streaming join started. Waiting for joined results...")
query.awaitTermination()

## Stopping The Stream

If you interrupted the previous cell, you can run this cleanup cell to stop the query gracefully.

In [ ]:
try:
    query.stop()
    print("Streaming query stopped.")
except NameError:
    print("No active query object found.")